# 🐼 Taller Práctico de Pandas
### Programación II — Módulo 3 · IUB Barranquilla

---

En este taller trabajaremos con un dataset real de **ventas de una tienda de tecnología** (`datos_ventas.xlsx`).  
Cada sección explica un concepto y luego propone una actividad para practicarlo.

**Columnas del dataset:**

| Columna | Descripción |
|---|---|
| `id_venta` | Identificador único de cada venta |
| `fecha` | Fecha de la venta (YYYY-MM-DD) |
| `vendedor` | Nombre del vendedor |
| `ciudad` | Ciudad donde se realizó la venta |
| `producto` | Nombre del producto vendido |
| `categoria` | Categoría del producto |
| `precio_unitario` | Precio por unidad (COP) |
| `cantidad` | Unidades vendidas |
| `total` | Total de la venta (`precio × cantidad`) |

---
## 0. Generar el archivo de ventas en diferentes formatos

Antes de empezar, **ejecuta la siguiente celda**. Genera el dataset de ventas de la tienda (220 ventas, enero–junio 2024) y lo guarda en tres formatos distintos:

- `datos_ventas.csv`
- `datos_ventas.parquet`
- `datos_ventas.xlsx` ← el que usaremos durante el taller

Al final imprime cuánto pesa cada archivo para compararlos.

In [ ]:
import os
import numpy as np
import pandas as pd

# --- Catálogo de productos: (producto, categoría, precio unitario en COP) ---
catalogo = [
    ('Laptop',              'Computadores',   2_850_000),
    ('Ultrabook',           'Computadores',   4_600_000),
    ('PC Escritorio',       'Computadores',   2_150_000),
    ('Smartphone',          'Moviles',        1_950_000),
    ('Tablet',              'Moviles',        1_180_000),
    ('Monitor',             'Monitores',        650_000),
    ('Monitor Curvo 32',    'Monitores',      1_450_000),
    ('Teclado Mecanico',    'Perifericos',      185_000),
    ('Mouse Inalambrico',   'Perifericos',       95_000),
    ('Audifonos Bluetooth', 'Perifericos',      260_000),
    ('Webcam HD',           'Perifericos',      145_000),
    ('SSD 1TB',             'Almacenamiento',   320_000),
    ('Disco Externo 2TB',   'Almacenamiento',   285_000),
    ('Memoria USB 128GB',   'Almacenamiento',    65_000),
    ('Cable HDMI',          'Accesorios',        38_000),
    ('Cargador USB-C',      'Accesorios',        58_000),
]
vendedores = ['Carlos Mejia', 'Ana Restrepo', 'Diana Torres',
              'Miguel Ospino', 'Jorge Salazar', 'Laura Pineda']
ciudades = ['Barranquilla', 'Bogota', 'Medellin', 'Cali', 'Cartagena']

rng = np.random.default_rng(42)   # semilla fija: todos obtienen los mismos datos
n = 220

idx = rng.integers(0, len(catalogo), n)
productos  = [catalogo[i][0] for i in idx]
categorias = [catalogo[i][1] for i in idx]
precios    = np.array([catalogo[i][2] for i in idx])

# Productos caros se venden en pocas unidades, los baratos en más
cantidades = np.where(precios >= 1_000_000,
                      rng.integers(1, 6, n),
                      rng.integers(1, 16, n)).astype(float)

fechas = pd.to_datetime('2024-01-01') + pd.to_timedelta(np.sort(rng.integers(0, 180, n)), unit='D')

ventas = pd.DataFrame({
    'id_venta':        [f'V{i:04d}' for i in range(1, n + 1)],
    'fecha':           fechas,
    'vendedor':        rng.choice(vendedores, n),
    'ciudad':          rng.choice(ciudades, n, p=[0.28, 0.24, 0.18, 0.16, 0.14]),
    'producto':        productos,
    'categoria':       categorias,
    'precio_unitario': precios,
    'cantidad':        cantidades,
})
ventas['total'] = (ventas['precio_unitario'] * ventas['cantidad']).astype(int)

# Datos faltantes a propósito (para practicar la sección 12)
ventas.loc[rng.choice(n, 9, replace=False), 'vendedor'] = np.nan
ventas.loc[rng.choice(n, 7, replace=False), 'cantidad'] = np.nan

# --- Guardar en los tres formatos ---
ventas.to_csv('datos_ventas.csv', index=False)
ventas.to_parquet('datos_ventas.parquet', index=False)
ventas.to_excel('datos_ventas.xlsx', sheet_name='Ventas', index=False)

for archivo in ['datos_ventas.csv', 'datos_ventas.parquet', 'datos_ventas.xlsx']:
    print(f"{archivo:<22} {os.path.getsize(archivo) / 1024:8.1f} KB")

ventas.head()

### ¿Qué es cada tipo de archivo?

### 📄 CSV (*Comma-Separated Values*) — `.csv`
- Archivo de **texto plano**: cada fila es una línea y las columnas se separan con comas (u otro separador como `;`).
- **Ventajas:** universal, se abre con cualquier editor de texto, Excel, bases de datos o cualquier lenguaje de programación. Fácil de inspeccionar y versionar.
- **Desventajas:** **no guarda los tipos de datos** (al leerlo, pandas los vuelve a adivinar: las fechas llegan como texto), no comprime, y es lento y pesado con muchos datos.
- **Uso típico:** intercambio sencillo de datos entre sistemas.

```python
df.to_csv('ventas.csv', index=False)
pd.read_csv('ventas.csv')
```

### 🧱 Parquet — `.parquet`
- Formato **binario y columnar** creado para Big Data (ecosistema Apache: Spark, Hadoop, etc.). Guarda los datos **por columnas** y no por filas.
- **Ventajas:** **conserva los tipos de datos** (enteros, fechas, booleanos…), está **comprimido** (ocupa mucho menos espacio) y es muy rápido de leer; además permite leer solo las columnas que necesitas.
- **Desventajas:** no se puede abrir con un editor de texto ni con Excel; requiere una librería como `pyarrow` o `fastparquet`.
- **Uso típico:** analítica de datos, data lakes, pipelines con grandes volúmenes.

```python
df.to_parquet('ventas.parquet', index=False)
pd.read_parquet('ventas.parquet', columns=['producto', 'total'])
```

### 📊 Excel — `.xlsx`
- Formato de **hoja de cálculo** de Microsoft Excel (internamente es un ZIP con archivos XML).
- **Ventajas:** muy conocido por usuarios no técnicos, admite **varias hojas** en un mismo archivo, formatos, colores y fórmulas.
- **Desventajas:** lento de leer/escribir, límite de ~1 millón de filas por hoja, y requiere una librería como `openpyxl`.
- **Uso típico:** reportes para compartir con personas de negocio.

```python
df.to_excel('ventas.xlsx', sheet_name='Ventas', index=False)
pd.read_excel('ventas.xlsx', sheet_name='Ventas')
```

### Resumen

| Formato | Tipo | ¿Guarda tipos de dato? | Tamaño | Velocidad | ¿Legible por humanos? |
|---|---|---|---|---|---|
| CSV | Texto plano | ❌ | Grande | Media | ✅ |
| Parquet | Binario columnar | ✅ | Pequeño (comprimido) | Muy rápida | ❌ |
| Excel | Hoja de cálculo | Parcial | Mediano | Lenta | ✅ (con Excel) |

Comprobemos si al volver a leer cada archivo se **conservan los tipos de dato** (fíjate en la columna `fecha`):

In [ ]:
pd.DataFrame({
    'original': ventas.dtypes,
    'csv':      pd.read_csv('datos_ventas.csv').dtypes,
    'parquet':  pd.read_parquet('datos_ventas.parquet').dtypes,
    'excel':    pd.read_excel('datos_ventas.xlsx').dtypes,
})

---
## 1. Importar librerías

Siempre importamos **pandas** con el alias `pd`. Es una convención universal que todos los programadores respetan.

También importamos **NumPy** con el alias `np` porque pandas lo usa internamente y a veces necesitamos sus funciones (como `np.nan`).

In [ ]:
import pandas as pd
import numpy as np

print('Pandas:', pd.__version__)
print('NumPy: ', np.__version__)

Pandas: 3.1.1
NumPy:  2.5.4


---
## 2. Cargar el archivo Excel — `read_excel()`

`pd.read_excel('archivo.xlsx')` lee un archivo de Excel y lo convierte en un **DataFrame** — la estructura tabular principal de pandas (filas × columnas).

Parámetros útiles:
- `sheet_name='Ventas'` → indica la hoja a leer (por defecto lee la primera)
- `index_col='id_venta'` → usa esa columna como índice de las filas

### 📌 Actividad
Carga el archivo `datos_ventas.xlsx` en un DataFrame llamado `df`. Luego muestra el DataFrame.

In [6]:
# Tu código aquí
df = pd.read_parquet("datos_ventas.parquet")
df.head()
type(df)

pandas.DataFrame

---
## 3. Exploración inicial — `head()`, `tail()`, `shape`

Cuando cargamos un dataset nuevo, lo primero es echar un vistazo rápido:

| Método / Atributo | Qué hace |
|---|---|
| `df.head(n)` | Muestra las primeras `n` filas (5 por defecto) |
| `df.tail(n)` | Muestra las últimas `n` filas |
| `df.shape` | Devuelve una tupla `(filas, columnas)` |
| `df.columns` | Lista con los nombres de las columnas |

### 📌 Actividad
1. Muestra las primeras **8 filas** del DataFrame.
2. Muestra las últimas **3 filas**.
3. Imprime cuántas filas y columnas tiene el dataset y los nombres de las columnas.

In [8]:
# 1. Tu código aquí
df.head(8)

,id_venta,fecha,vendedor,ciudad,producto,categoria,precio_unitario,cantidad,total
0,V0001,2024-01-01,Carlos Mejia,Bogota,Ultrabook,Computadores,4600000,1.0,4600000
1,V0002,2024-01-01,Laura Pineda,Barranquilla,Disco Externo 2TB,Almacenamiento,285000,5.0,1425000
2,V0003,2024-01-01,Diana Torres,Barranquilla,Webcam HD,Perifericos,145000,2.0,290000
3,V0004,2024-01-02,Carlos Mejia,Cartagena,Teclado Mecanico,Perifericos,185000,8.0,1480000
4,V0005,2024-01-03,Jorge Salazar,Barranquilla,Monitor Curvo 32,Monitores,1450000,4.0,5800000
5,V0006,2024-01-03,Jorge Salazar,Barranquilla,Memoria USB 128GB,Almacenamiento,65000,10.0,650000
6,V0007,2024-01-03,Carlos Mejia,Bogota,Ultrabook,Computadores,4600000,5.0,23000000
7,V0008,2024-01-05,Ana Restrepo,Bogota,SSD 1TB,Almacenamiento,320000,15.0,4800000


In [10]:
# 2. Tu código aquí
df.tail(3)

,id_venta,fecha,vendedor,ciudad,producto,categoria,precio_unitario,cantidad,total
217,V0218,2024-06-28,Jorge Salazar,Barranquilla,Ultrabook,Computadores,4600000,5.0,23000000
218,V0219,2024-06-28,Carlos Mejia,Barranquilla,Teclado Mecanico,Perifericos,185000,2.0,370000
219,V0220,2024-06-28,Diana Torres,Medellin,Cable HDMI,Accesorios,38000,5.0,190000


In [11]:
# 3. Tu código aquí
df.shape

(220, 9)

In [ ]:
df.columns=['id_venta_2', 'fecha_"', 'vendedor', 'ciudad', 'producto', 'categoria',
       'precio_unitario', 'cantidad', 'total']

Index(['id_venta', 'fecha', 'vendedor', 'ciudad', 'producto', 'categoria',
       'precio_unitario', 'cantidad', 'total'],
      dtype='str')

---
## 4. Información general — `info()` y `dtypes`

`df.info()` muestra el **tipo de dato** de cada columna, cuántos **valores no nulos** tiene, y cuánta **memoria** ocupa el DataFrame.

Tipos de datos más comunes en pandas:
- `int64` → enteros
- `float64` → decimales (o enteros con NaN)
- `object` → texto (strings)
- `datetime64` → fechas
- `bool` → verdadero/falso

### 📌 Actividad
1. Ejecuta `df.info()` e identifica qué columnas tienen valores faltantes.
2. Imprime solo los tipos de dato con `df.dtypes`.

In [17]:
# 1. Tu código aquí
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_venta         220 non-null    str           
 1   fecha            220 non-null    datetime64[us]
 2   vendedor         211 non-null    str           
 3   ciudad           220 non-null    str           
 4   producto         220 non-null    str           
 5   categoria        220 non-null    str           
 6   precio_unitario  220 non-null    int64         
 7   cantidad         213 non-null    float64       
 8   total            220 non-null    int64         
dtypes: datetime64[us](1), float64(1), int64(2), str(5)
memory usage: 15.6 KB


In [19]:
# 2. Tu código aquí
df.dtypes

id_venta                      str
fecha              datetime64[us]
vendedor                      str
ciudad                        str
producto                      str
categoria                     str
precio_unitario             int64
cantidad                  float64
total                       int64
dtype: object

---
## 5. Estadísticas descriptivas — `describe()`

`df.describe()` calcula automáticamente las estadísticas más importantes de las **columnas numéricas**:
- `count` → cantidad de valores no nulos
- `mean` → promedio
- `std` → desviación estándar
- `min` / `max` → mínimo y máximo
- `25%`, `50%`, `75%` → percentiles (cuartiles)

### 📌 Actividad
1. Ejecuta `df.describe()` sobre todo el DataFrame.
2. Calcula el **total de ventas** (suma de la columna `total`) e imprímelo con formato de miles.
3. ¿Cuál es el precio unitario **promedio**? ¿Y el **máximo**?

In [20]:
# 1. Tu código aquí
df.describe()

,fecha,precio_unitario,cantidad,total
count,220,2.200000e+02,213.000000,2.200000e+02
mean,2024-04-02 21:09:49.090909,9.707864e+05,6.455399,3.701827e+06
min,2024-01-01 00:00:00,3.800000e+04,1.000000,3.800000e+04
25%,2024-02-06 18:00:00,1.450000e+05,3.000000,8.525000e+05
50%,2024-04-10 00:00:00,2.850000e+05,5.000000,2.080000e+06
75%,2024-05-20 00:00:00,1.450000e+06,10.000000,4.600000e+06
max,2024-06-28 00:00:00,4.600000e+06,15.000000,2.300000e+07
std,NaN,1.295704e+06,4.460956,4.533537e+06


In [22]:
# 2. Tu código aquí
df['total'].sum()

np.int64(814402000)

In [24]:
# 3. Tu código aquí
print(df['precio_unitario'].mean(),df['precio_unitario'].max(),sep="\n")

970786.3636363636
4600000


---
## 6. Selección de columnas y filas — `loc` e `iloc`

### Seleccionar columnas
```python
df['columna']           # devuelve una Series
df[['col1', 'col2']]    # devuelve un DataFrame
```

### `loc` — por etiqueta (nombre de columna / índice)
```python
df.loc[fila, columna]              # celda específica
df.loc[0:4, 'vendedor':'total']    # rango (¡incluye el extremo final!)
```

### `iloc` — por posición numérica (como listas de Python)
```python
df.iloc[0]          # primera fila
df.iloc[0:5, 0:3]   # filas 0-4, columnas 0-2
```

### 📌 Actividad
1. Selecciona solo las columnas `vendedor`, `producto` y `total`.
2. Usa `loc` para obtener las filas 10 a 14 (solo columnas `fecha`, `ciudad`, `total`).
3. Usa `iloc` para obtener la última fila del DataFrame.

In [26]:
# 1. Tu código aquí
df['vendedor']

0       Carlos Mejia
1       Laura Pineda
2       Diana Torres
3       Carlos Mejia
4      Jorge Salazar
           ...      
215     Carlos Mejia
216     Laura Pineda
217    Jorge Salazar
218     Carlos Mejia
219     Diana Torres
Name: vendedor, Length: 220, dtype: str

In [31]:
df[['vendedor','producto','categoria']].head()

,vendedor,producto,categoria
0,Carlos Mejia,Ultrabook,Computadores
1,Laura Pineda,Disco Externo 2TB,Almacenamiento
2,Diana Torres,Webcam HD,Perifericos
3,Carlos Mejia,Teclado Mecanico,Perifericos
4,Jorge Salazar,Monitor Curvo 32,Monitores


In [35]:
# 2. Tu código aquí
df.loc[0:18,'id_venta':'ciudad'].head()

,id_venta,fecha,vendedor,ciudad
0,V0001,2024-01-01,Carlos Mejia,Bogota
1,V0002,2024-01-01,Laura Pineda,Barranquilla
2,V0003,2024-01-01,Diana Torres,Barranquilla
3,V0004,2024-01-02,Carlos Mejia,Cartagena
4,V0005,2024-01-03,Jorge Salazar,Barranquilla


In [38]:
# 3. Tu código aquí
df.iloc[0:18,0:4]

,id_venta,fecha,vendedor,ciudad
0,V0001,2024-01-01,Carlos Mejia,Bogota
1,V0002,2024-01-01,Laura Pineda,Barranquilla
2,V0003,2024-01-01,Diana Torres,Barranquilla
3,V0004,2024-01-02,Carlos Mejia,Cartagena
4,V0005,2024-01-03,Jorge Salazar,Barranquilla
5,V0006,2024-01-03,Jorge Salazar,Barranquilla
6,V0007,2024-01-03,Carlos Mejia,Bogota
7,V0008,2024-01-05,Ana Restrepo,Bogota
8,V0009,2024-01-06,Carlos Mejia,Medellin
9,V0010,2024-01-06,Ana Restrepo,Bogota


---
## 7. Filtrado de datos

El filtrado usa **condiciones booleanas** dentro de corchetes. Pandas evalúa la condición fila por fila y solo devuelve las que son `True`.

```python
df[df['columna'] > valor]                          # una condición
df[(df['col1'] > x) & (df['col2'] == 'texto')]     # AND  →  &
df[(df['col1'] > x) | (df['col2'] == 'texto')]     # OR   →  |
df[~(df['col'] == 'texto')]                         # NOT  →  ~
```

> ⚠️ Siempre encierra cada condición en paréntesis cuando uses `&` o `|`.

### 📌 Actividad
1. Filtra las ventas cuyo `total` sea mayor a **5,000,000 COP**.
2. Filtra las ventas de **Barranquilla** que sean de la categoría **Computadores**.
3. Usa `.isin()` para filtrar las ventas de **Laptop** o **Monitor**.
4. Usa `.query()` para obtener las ventas donde `cantidad >= 8`.

In [ ]:
# 1. Tu código aquí

In [ ]:
# 2. Tu código aquí

In [ ]:
# 3. Tu código aquí

In [ ]:
# 4. Tu código aquí

---
## 8. Ordenar datos — `sort_values()`

`sort_values()` ordena el DataFrame por una o más columnas.

```python
df.sort_values('columna')                                      # ascendente
df.sort_values('columna', ascending=False)                     # descendente
df.sort_values(['col1', 'col2'], ascending=[True, False])      # múltiples
```

### 📌 Actividad
1. Ordena el DataFrame por `total` de mayor a menor.
2. Ordena por `ciudad` (A→Z) y dentro de cada ciudad por `total` (mayor a menor).
3. Muestra el **top 5** de ventas más altas (solo columnas `fecha`, `vendedor`, `producto`, `total`).

In [ ]:
# 1. Tu código aquí

In [ ]:
# 2. Tu código aquí

In [ ]:
# 3. Tu código aquí

---
## 9. Conteos y valores únicos — `value_counts()`, `unique()`, `nunique()`

Estos métodos se aplican sobre una **Series** (una columna):

| Método | Qué devuelve |
|---|---|
| `.value_counts()` | Frecuencia de cada valor, ordenado de mayor a menor |
| `.unique()` | Array con los valores únicos |
| `.nunique()` | Cantidad de valores únicos (un solo número) |

### 📌 Actividad
1. ¿Cuántas ventas hizo cada **vendedor**? Usa `value_counts()`.
2. ¿Qué **productos** distintos aparecen en el dataset? Usa `unique()`.
3. ¿Cuántas **ciudades** diferentes hay? Usa `nunique()`.

In [ ]:
# 1. Tu código aquí

In [ ]:
# 2. Tu código aquí

In [ ]:
# 3. Tu código aquí

---
## 10. Agrupaciones — `groupby()`

`groupby()` agrupa las filas por los valores únicos de una columna y permite aplicar una función de agregación a cada grupo.

```python
df.groupby('columna')['otra_columna'].mean()              # promedio por grupo
df.groupby('columna').agg({'col1': 'sum', 'col2': 'mean'})  # varias funciones

# Con nombres personalizados para las columnas resultantes:
df.groupby('col')['val'].agg(total='sum', promedio='mean', conteo='count')
```

### 📌 Actividad
1. Calcula el **total de ventas por ciudad** (ordenado de mayor a menor).
2. Calcula el **promedio de cantidad vendida por categoría**.
3. Usa `.agg()` para obtener — por vendedor — la **suma**, **promedio** y **cantidad** de ventas.
4. ¿Qué **producto** generó más ingresos en total?

In [ ]:
# 1. Tu código aquí

In [ ]:
# 2. Tu código aquí

In [ ]:
# 3. Tu código aquí

In [ ]:
# 4. Tu código aquí

---
## 11. Agregar columnas calculadas

Podemos crear nuevas columnas a partir de operaciones sobre columnas existentes. Pandas aplica la operación **fila por fila** de forma automática (vectorización).

```python
df['nueva_columna'] = df['col1'] * df['col2']
df['nueva_columna'] = df['col'].apply(lambda x: 'alto' if x > 100 else 'bajo')
```

### 📌 Actividad
1. Crea una columna `iva` que sea el **19%** del `total`.
2. Crea una columna `total_con_iva` sumando `total` + `iva`.
3. Crea una columna `es_premium` que sea `True` si el `precio_unitario` supera **500,000 COP**.
4. Muestra las columnas `producto`, `total`, `iva`, `total_con_iva`, `es_premium` de las primeras 8 filas.

In [ ]:
# Tu código aquí

---
## 12. Datos faltantes — `isnull()`, `dropna()`, `fillna()`

En datasets reales siempre hay valores faltantes, representados como **NaN** (_Not a Number_).

| Método | Qué hace |
|---|---|
| `df.isnull()` | DataFrame de `True`/`False` indicando dónde hay NaN |
| `df.isnull().sum()` | Cuenta NaN por columna |
| `df.dropna()` | Elimina filas con algún NaN |
| `df.fillna(valor)` | Rellena NaN con un valor dado |

> `fillna()` y `dropna()` no modifican el DataFrame original a menos que uses `inplace=True` o reasignes: `df = df.dropna()`.

### 📌 Actividad
1. ¿Cuántos valores faltantes hay en cada columna?
2. Rellena los NaN de `vendedor` con el texto `'Sin asignar'`.
3. Rellena los NaN de `cantidad` con el **promedio** de esa columna (redondeado).
4. Verifica que ya no quedan NaN en esas columnas.

In [ ]:
# 1. Tu código aquí

In [ ]:
# 2. Tu código aquí

In [ ]:
# 3. Tu código aquí

In [ ]:
# 4. Tu código aquí

---
## 13. Guardar resultados — `to_csv()`, `to_excel()`

Después de procesar el dataset podemos exportar el resultado a un nuevo archivo.

```python
df.to_csv('resultado.csv', index=False)                          # CSV sin índice
df.to_excel('resultado.xlsx', sheet_name='Hoja1', index=False)  # Excel
```

> `index=False` evita que pandas guarde la columna de índice (0, 1, 2…) como una columna extra.

### 📌 Actividad
1. Guarda el DataFrame completo (ya limpio) como `ventas_procesadas.csv`.
2. Crea un DataFrame con el resumen de ventas por vendedor (sección 10) y guárdalo como `resumen_vendedores.xlsx`.

In [ ]:
# 1. Tu código aquí

In [ ]:
# 2. Tu código aquí

---
## 🏁 Reto final

Usando todo lo aprendido, responde las siguientes preguntas. Una celda por pregunta:

1. **¿Qué vendedor generó el mayor ingreso total?**
2. **¿Cuántas ventas de productos premium (`precio_unitario > 500,000`) se hicieron en Bogotá?**
3. **¿Cuál es el mes con más ventas?** *(pista: convierte `fecha` a datetime con `pd.to_datetime()` y usa `.dt.month`)*
4. **Crea un DataFrame de resumen** con columnas `ciudad`, `total_ventas` y `num_transacciones`, ordenado por `total_ventas` descendente.

In [ ]:
# Reto 1

In [ ]:
# Reto 2

In [ ]:
# Reto 3

In [ ]:
# Reto 4